# Reader and Writer Packages


In [ ]:
# Import the regular expression module from the standard library
import re

In [ ]:
def address_to_indices(address):
    """Convert A1 notation to a zero-indexed tuple (row, col)."""
    # Use regex to split e.g., "B10" into "B" and "10"
    match = re.match(r"([A-Z]+)(\d+)", address.upper())
    col_letters, row_str = match.groups()
    # Convert column letters to a number (A=1, ..., Z=26, AA=27, etc.)
    col_num = 0
    for char in col_letters:
        col_num = col_num * 26 + (ord(char) - ord("A") + 1)
    # Return as zero-indexed tuple
    return int(row_str) - 1, col_num - 1

## python-calamine


In [ ]:
from python_calamine import CalamineWorkbook

In [ ]:
values = (
    CalamineWorkbook.from_path("xl/stores.xlsx")
    .get_sheet_by_name("2019")
    .to_python(skip_empty_area=True)
)
values[:3]

## fastexcel


In [ ]:
import fastexcel

In [ ]:
# Requires PyArrow
reader = fastexcel.read_excel("xl/stores.xlsx")
sheet = reader.load_sheet("2019")
df = sheet.to_pandas()
df.head(2)

In [ ]:
reader = fastexcel.read_excel("xl/stores.xlsx")
reader.table_names()

In [ ]:
# Requires PyArrow
table2019 = reader.load_table("Table2020").to_pandas()
table2019.head(2)

## XlsxWriter


In [ ]:
import datetime as dt
import xlsxwriter

In [ ]:
# Helper function to write a nested list with XlsxWriter
def write_xlsxwriter(sheet, values, first_cell="A1"):
    first_cell = address_to_indices(first_cell)

    for ix, row_data in enumerate(values):
        sheet.write_row(first_cell[0] + ix, first_cell[1], row_data)

In [ ]:
# Instantiate a workbook
book1 = xlsxwriter.Workbook("xlsxwriter.xlsx")

# Add a sheet and give it a name
sheet = book1.add_worksheet("Sheet1")

# Writing individual cells using A1 notation
# and cell indices (0-based)
sheet.write("A1", "Hello 1")
sheet.write(1, 0, "Hello 2")

# Formatting: fill color, alignment, border and font
formatting = book1.add_format(
    {
        "font_color": "#FF0000",
        "bg_color": "#FFFF00",
        "bold": True,
        "align": "center",
        "border": 1,
        "border_color": "#FF0000",
    }
)
sheet.write("A3", "Hello 3", formatting)

# Number formatting (using Excel's formatting strings)
number_format = book1.add_format({"num_format": "0.00"})
sheet.write("A4", 3.3333, number_format)

# Date formatting (using Excel's formatting strings)
date_format = book1.add_format({"num_format": "mm/dd/yy"})
sheet.write("A5", dt.date(2016, 10, 13), date_format)

# Formula: you must use the English name of the formula
# with commas as delimiters
sheet.write("A6", "=SUM(A4, 2)")

# Image
sheet.insert_image(0, 2, "images/python.png")

# Two-dimensional list
data = [
    [None, "North", "South"],
    ["Last Year", 2, 5],
    ["This Year", 3, 6],
]
write_xlsxwriter(sheet=sheet, values=data, first_cell="A10")

# Chart: see the file "sales_report_xlsxwriter.py" in the
# companion repo to see how you can work with indices
# instead of cell addresses
chart = book1.add_chart({"type": "column"})
chart.set_title({"name": "Sales per Region"})
chart.add_series(
    {
        "name": "=Sheet1!A11",
        "categories": "=Sheet1!B10:C10",
        "values": "=Sheet1!B11:C11",
    }
)
chart.add_series(
    {
        "name": "=Sheet1!A12",
        "categories": "=Sheet1!B10:C10",
        "values": "=Sheet1!B12:C12",
    }
)
chart.set_x_axis({"name": "Regions"})
chart.set_y_axis({"name": "Sales"})
sheet.insert_chart("A15", chart)

# Closing the workbook creates the file on disk
book1.close()

In [ ]:
book2 = xlsxwriter.Workbook("macro_xlsxwriter.xlsm")
sheet = book2.add_worksheet("Sheet1")
sheet.write("A1", "Click the button!")
book2.add_vba_project("xl/vbaProject.bin")
sheet.insert_button(
    "A3",
    {
        "macro": "Hello",
        "caption": "Button 1",
        "width": 130,
        "height": 35,
    },
)
book2.close()

In [ ]:
import random

book3 = xlsxwriter.Workbook(
    "xlsxwriter_optimized.xlsx", options={"constant_memory": True}
)
sheet = book3.add_worksheet()
# This will produce a sheet with 1000 x 200 cells.
# With constant_memory, you must write data in row by row, i.e.,
# you first need to loop through rows and then through columns.
for row in range(1000):
    for col in range(200):
        sheet.write(row, col, random.random())
book3.close()

## OpenPyXL

### Reading with OpenPyXL


In [ ]:
import datetime as dt
import openpyxl
import pandas as pd


In [ ]:
# Open the workbook to read cell values.
# The file is automatically closed again after loading the data.
book4 = openpyxl.load_workbook("xl/stores.xlsx", data_only=True)

In [ ]:
# Get a worksheet object by name or index (0-based)
sheet = book4["2019"]
sheet = book4.worksheets[0]

In [ ]:
# Get a list with all sheet names
book4.sheetnames

In [ ]:
# Loop through the sheet objects.
# Instead of "name", openpyxl uses "title".
for row_ix in book4.worksheets:
    print(row_ix.title)

In [ ]:
# Getting the dimensions,
# i.e., the used range of the sheet
sheet.max_row, sheet.max_column

In [ ]:
# Read the value of a single cell
# using "A1" notation and using cell indices (1-based)
sheet["B6"].value
sheet.cell(row=6, column=2).value

In [ ]:
# Helper function to read an Excel range with OpenPyXL
def read_openpyxl(sheet, first_cell="A1", last_cell=None):
    first_cell = openpyxl.utils.cell.coordinate_to_tuple(first_cell)
    if last_cell is None:
        last_cell = (sheet.max_row, sheet.max_column)
    else:
        last_cell = openpyxl.utils.cell.coordinate_to_tuple(last_cell)

    data = []
    for row in sheet.iter_rows(
        min_row=first_cell[0],
        min_col=first_cell[1],
        max_row=last_cell[0],
        max_col=last_cell[1],
        values_only=True,
    ):
        data.append(list(row))
    return data

In [ ]:
# Read in a range of cell values
data = read_openpyxl(
    sheet=book4["2019"],
    first_cell="B2",
    last_cell="F8",
)
data[:2]  # Print the first two rows

In [ ]:
book5 = openpyxl.load_workbook(
    "xl/big.xlsx", data_only=True, read_only=True, keep_links=False
)
# Perform the desired read operations here
book5.close()  # Required with read_only=True

### Writing with OpenPyXL


In [ ]:
import openpyxl
from openpyxl.chart import BarChart, Reference
from openpyxl.drawing.image import Image
from openpyxl.styles import Font
from openpyxl.styles.alignment import Alignment
from openpyxl.styles.borders import Border, Side
from openpyxl.styles.fills import PatternFill


In [ ]:
# Unfortunately, the latest version of OpenPyXL has an issue with
# chart formatting. This patch sets the Application property to fix it.
from openpyxl.packaging.extended import ExtendedProperties

_original_init = ExtendedProperties.__init__


def patched_init(self, *args, **kwargs):
    _original_init(self, *args, **kwargs)
    self.Application = "Microsoft Excel"


ExtendedProperties.__init__ = patched_init

In [ ]:
# Helper function to write a nested list with OpenPyXL
def write_openpyxl(
    sheet,
    values,
    first_cell="A1",
    date_format="mm/dd/yy",
):
    first_cell = openpyxl.utils.coordinate_to_tuple(first_cell)

    for row_ix, row in enumerate(values):
        for col_ix, value in enumerate(row):
            cell = sheet.cell(
                row=first_cell[0] + row_ix,
                column=first_cell[1] + col_ix,
            )
            cell.value = value
            # Format dates
            if isinstance(value, (dt.datetime, dt.date)):
                cell.number_format = date_format

In [ ]:
# Instantiate a workbook
book6 = openpyxl.Workbook()
book6.properties.application = "Microsoft Excel"

# Get the first sheet and give it a name
sheet = book6.active
sheet.title = "Sheet1"

# Writing individual cells using A1 notation
# and cell indices (1-based)
sheet["A1"].value = "Hello 1"
sheet.cell(row=2, column=1, value="Hello 2")

# Formatting: fill color, alignment, border and font
font_format = Font(color="FF0000", bold=True)
thin = Side(border_style="thin", color="FF0000")
sheet["A3"].value = "Hello 3"
sheet["A3"].font = font_format
sheet["A3"].border = Border(
    top=thin,
    left=thin,
    right=thin,
    bottom=thin,
)
sheet["A3"].alignment = Alignment(horizontal="center")
sheet["A3"].fill = PatternFill(fgColor="FFFF00", fill_type="solid")

# Number formatting (using Excel's formatting strings)
sheet["A4"].value = 3.3333
sheet["A4"].number_format = "0.00"

# Date formatting (using Excel's formatting strings)
sheet["A5"].value = dt.date(2016, 10, 13)
sheet["A5"].number_format = "mm/dd/yy"

# Formula: you must use the English name of the formula
# with commas as delimiters
sheet["A6"].value = "=SUM(A4, 2)"

# Image
sheet.add_image(Image("images/python.png"), "C1")

# Two-dimensional list
data = [
    [None, "North", "South"],
    ["Last Year", 2, 5],
    ["This Year", 3, 6],
]
write_openpyxl(sheet=sheet, values=data, first_cell="A10")

# Chart
chart = BarChart()
chart.type = "col"
chart.title = "Sales Per Region"
chart.x_axis.title = "Regions"
chart.y_axis.title = "Sales"
chart_data = Reference(
    sheet,
    min_row=11,
    min_col=1,
    max_row=12,
    max_col=3,
)
chart_categories = Reference(
    sheet,
    min_row=10,
    min_col=2,
    max_row=10,
    max_col=3,
)
# from_rows interprets the data in the same way
# as if you would add a chart manually in Excel
chart.add_data(chart_data, titles_from_data=True, from_rows=True)
chart.set_categories(chart_categories)
sheet.add_chart(chart, "A15")

# Saving the workbook creates the file on disk
book6.save("openpyxl.xlsx")

In [ ]:
book7 = openpyxl.Workbook(write_only=True)
# With write_only=True, book.active doesn't work
sheet = book7.create_sheet()
# This will produce a sheet with 1000 x 200 cells
for row in range(1000):
    sheet.append(list(range(200)))
book7.save("openpyxl_optimized.xlsx")

### Editing with OpenPyXL


In [ ]:
# Read the stores.xlsx file, change a cell
# and store it under a new location/name.
book8 = openpyxl.load_workbook("xl/stores.xlsx")
book8["2019"]["A1"].value = "modified"
book8.save("stores_edited.xlsx")

In [ ]:
book9 = openpyxl.load_workbook("xl/macro.xlsm", keep_vba=True)
book9["Sheet1"]["A1"].value = "Click the button!"
book9.save("macro_openpyxl.xlsm")

## Excelize


In [ ]:
import datetime as dt
import excelize

In [ ]:
book10 = excelize.new_file()
data = [
    ["Item", "Quantity", "Price"],
    ["Apple", 10, 1.20],
    ["Banana", 15, 0.75],
    ["Orange", 8, 1.50],
    ["Apple", 12, 1.25],
]

for row_ix, row_data in enumerate(data):
    book10.set_sheet_row("Sheet1", f"A{row_ix + 1}", row_data)

num_data_rows = len(data)

book10.add_pivot_table(
    excelize.PivotTableOptions(
        data_range="Sheet1!A1:C5",
        pivot_table_range="Sheet1!E1:G6",
        rows=[excelize.PivotTableField(data="Item")],
        data=[
            excelize.PivotTableField(
                data="Quantity", name="Total Quantity", subtotal="Sum"
            ),
            excelize.PivotTableField(
                data="Price", name="Average Price", subtotal="Average"
            ),
        ],
        row_grand_totals=True,
        col_grand_totals=True,
    )
)
book10.save_as("excelize_pivot.xlsx")
book10.close()

In [ ]:
book11 = excelize.new_file()

In [ ]:
sheet = book11.new_sheet("Sheet1")
book11.set_active_sheet(sheet)

In [ ]:
# Writing individual cells using A1 notation
# and cell indices (1-based)
book11.set_cell_value("Sheet1", "A1", "Hello 1")
book11.set_cell_value(
    "Sheet1",
    excelize.coordinates_to_cell_name(row=2, col=1),
    "Hello2",
)

In [ ]:
# Formatting: fill color, alignment, border and font
style_id = book11.new_style(
    excelize.Style(
        font=excelize.Font(
            color="FF0000",
            bold=True,
        ),
        border=[
            excelize.Border(type="left", color="FF0000", style=1),
            excelize.Border(type="right", color="FF0000", style=1),
            excelize.Border(type="top", color="FF0000", style=1),
            excelize.Border(type="bottom", color="FF0000", style=1),
        ],
        fill=excelize.Fill(
            type="pattern",
            pattern=1,
            color=["FFFF00"],
        ),
        alignment=excelize.Alignment(
            horizontal="center",
        ),
    )
)

# Apply the style to cell A3
book11.set_cell_value("Sheet1", "A3", "Hello 3")
book11.set_cell_style("Sheet1", "A3", "A3", style_id)

In [ ]:
# Number formatting (using Excel's formatting strings)
book11.set_cell_value("Sheet1", "A4", 3.3333)
style_id = book11.new_style(
    excelize.Style(
        custom_num_fmt="0.00",
    )
)
book11.set_cell_style("Sheet1", "A4", "A4", style_id)

In [ ]:
# Date formatting (using Excel's formatting strings)
book11.set_cell_value("Sheet1", "A5", dt.date(2016, 10, 13))
style_id = book11.new_style(
    excelize.Style(
        custom_num_fmt="mm/dd/yy",
    )
)
book11.set_cell_style("Sheet1", "A5", "A5", style_id)

In [ ]:
# Formula: you must use the English name of the formula
# with commas as delimiters
book11.set_cell_formula("Sheet1", "A6", "=SUM(A4, 2)")

In [ ]:
# Image
book11.add_picture(
    "Sheet1",
    "C1",
    "images/python.png",
    excelize.GraphicOptions(),
)

In [ ]:
# Helper function to write a nested list with excelize
def write_excelize(book, sheet_name, values, first_cell="A1"):
    first_col, first_row = excelize.cell_name_to_coordinates(first_cell)

    for ix, row in enumerate(values):
        cell = excelize.coordinates_to_cell_name(row=first_row + ix, col=first_col)
        book.set_sheet_row(sheet_name, cell, row)

In [ ]:
# Two-dimensional list
data = [
    [None, "North", "South"],
    ["Last Year", 2, 5],
    ["This Year", 3, 6],
]
write_excelize(book11, "Sheet1", data, "A10")

In [ ]:
# Chart
chart = excelize.Chart(
    type=excelize.ChartType.Col,
    series=[
        excelize.ChartSeries(
            name="Sheet1!$A$11",
            categories="Sheet1!$B$10:$C$10",
            values="Sheet1!$B$11:$C$11",
        ),
        excelize.ChartSeries(
            name="Sheet1!$A$12",
            categories="Sheet1!$B$10:$C$10",
            values="Sheet1!$B$12:$C$12",
        ),
    ],
    title=[excelize.RichTextRun(text="Sales per Region")],
)
book11.add_chart("Sheet1", "A15", chart)

In [ ]:
book11.save_as("excelize.xlsx")

## pyxlsb


In [ ]:
import itertools
import pyxlsb

In [ ]:
# Helper function to read an Excel range with pyxlsb
def read_pyxlsb(sheet, first_cell="A1", last_cell=None):
    errors = {
        "0x0": "#NULL!",
        "0x7": "#DIV/0!",
        "0xf": "#VALUE!",
        "0x17": "#REF!",
        "0x1d": "#NAME?",
        "0x24": "#NUM!",
        "0x2a": "#N/A",
    }

    first_cell = address_to_indices(first_cell)
    first_cell = (first_cell[0] + 1, first_cell[1] + 1)

    if last_cell:
        last_cell = address_to_indices(last_cell)
        last_cell = (last_cell[0] + 1, last_cell[1] + 1)

    data = []
    # sheet.rows() is a generator that requires islice to slice it
    for row in itertools.islice(
        sheet.rows(),
        first_cell[0] - 1,
        last_cell[0] if last_cell else None,
    ):
        data.append(
            [errors.get(cell.v, cell.v) for cell in row][
                first_cell[1] - 1 : last_cell[1] if last_cell else None
            ]
        )
    return data

In [ ]:
# Loop through sheets. With pyxlsb, the workbook
# and sheet objects can be used as context managers.
# book.sheets returns a list of sheet names, not objects!
# To get a sheet object, use get_sheet() instead.
with pyxlsb.open_workbook("xl/stores.xlsb") as book12:
    for sheet_name in book12.sheets:
        with book12.get_sheet(sheet_name) as sheet:
            dim = sheet.dimension
            print(f"'{sheet_name}' has {dim.h} rows and {dim.w} cols")

In [ ]:
# Read in the values of a range of cells.
# Instead of "2019", you could also use its index (1-based).
with pyxlsb.open_workbook("xl/stores.xlsb") as book13:
    with book13.get_sheet("2019") as sheet:
        data = read_pyxlsb(sheet, "B2")
data[:2]  # Print the first two rows

In [ ]:
from pyxlsb import convert_date

convert_date(data[1][3])

## xlrd, xlwt and xlutils


### Reading with xlrd


In [ ]:
import xlrd
from xlrd.biffh import error_text_from_code
from xlwt.Utils import cell_to_rowcol2

In [ ]:
# Open the workbook to read cell values. The file is
# automatically closed again after loading the data.
book14 = xlrd.open_workbook("xl/stores.xls")

In [ ]:
# Get a list with all sheet names
book14.sheet_names()

In [ ]:
# Loop through the sheet objects
for sheet in book14.sheets():
    print(sheet.name)

In [ ]:
# Get a sheet object by name or index (0-based)
sheet = book14.sheet_by_index(0)
sheet = book14.sheet_by_name("2019")

In [ ]:
# Dimensions
sheet.nrows, sheet.ncols

In [ ]:
# Read the value of a single cell
# using "A1" notation and using cell indices (0-based).
# The "*" unpacks the tuple that cell_to_rowcol2 returns
# into individual arguments.
sheet.cell(*cell_to_rowcol2("B3")).value
sheet.cell(2, 1).value

In [ ]:
# Helper function to read an Excel range with xlrd
def read_xlrd(sheet, first_cell="A1", last_cell=None):
    if last_cell is None:
        last_cell = (sheet.nrows, sheet.ncols)
    else:
        last_cell = address_to_indices(last_cell)
        last_cell = (last_cell[0] + 1, last_cell[1] + 1)

    first_cell = address_to_indices(first_cell)
    first_cell = (first_cell[0] + 1, first_cell[1] + 1)

    values = []
    for r in range(first_cell[0] - 1, last_cell[0]):
        row = []
        for c in range(first_cell[1] - 1, last_cell[1]):
            if sheet.cell(r, c).ctype == xlrd.XL_CELL_DATE:
                value = xlrd.xldate.xldate_as_datetime(
                    sheet.cell(r, c).value, sheet.book.datemode
                )
            elif sheet.cell(r, c).ctype in [
                xlrd.XL_CELL_EMPTY,
                xlrd.XL_CELL_BLANK,
            ]:
                value = None
            elif sheet.cell(r, c).ctype == xlrd.XL_CELL_ERROR:
                value = error_text_from_code[sheet.cell(r, c).value]
            elif sheet.cell(r, c).ctype == xlrd.XL_CELL_BOOLEAN:
                value = bool(sheet.cell(r, c).value)
            else:
                value = sheet.cell(r, c).value
            row.append(value)
        values.append(row)
    return values

In [ ]:
# Read in a range of cell values
data = read_xlrd(sheet=sheet, first_cell="B2")
data[:2]  # Print the first two rows

In [ ]:
with xlrd.open_workbook("xl/stores.xls", on_demand=True) as book15:
    sheet = book15.sheet_by_index(0)  # Only loads the first sheet

In [ ]:
with xlrd.open_workbook("xl/stores.xls", on_demand=True) as book16:
    with pd.ExcelFile(book16, engine="xlrd") as f:
        df = pd.read_excel(f, sheet_name=0)

### Writing with xlwt


In [ ]:
import datetime as dt
import xlwt
from xlwt.Utils import cell_to_rowcol2

In [ ]:
# Helper function to write a nested list with xlwt
def write_xlwt(sheet, values, first_cell="A1", date_format="mm/dd/yy"):
    date_format = xlwt.easyxf(num_format_str=date_format)
    first_cell = address_to_indices(first_cell)

    for row_ix, row in enumerate(values):
        for col_ix, cell in enumerate(row):
            if isinstance(cell, (dt.datetime, dt.date)):
                sheet.write(
                    row_ix + first_cell[0],
                    col_ix + first_cell[1],
                    cell,
                    date_format,
                )
            else:
                sheet.write(
                    row_ix + first_cell[0],
                    col_ix + first_cell[1],
                    cell,
                )

In [ ]:
# Instantiate a workbook
book17 = xlwt.Workbook()

# Add a sheet and give it a name
sheet = book17.add_sheet("Sheet1")

# Writing individual cells using A1 notation
# and cell indices (0-based)
sheet.write(*cell_to_rowcol2("A1"), "Hello 1")
sheet.write(r=1, c=0, label="Hello 2")

# Formatting: fill color, alignment, border and font
formatting = xlwt.easyxf(
    "font: bold on, color red;"
    "align: horiz center;"
    "borders: top_color red, bottom_color red,"
    "right_color red, left_color red,"
    "left thin, right thin,"
    "top thin, bottom thin;"
    "pattern: pattern solid, fore_color yellow;"
)
sheet.write(r=2, c=0, label="Hello 3", style=formatting)

# Number formatting (using Excel's formatting strings)
number_format = xlwt.easyxf(num_format_str="0.00")
sheet.write(3, 0, 3.3333, number_format)

# Date formatting (using Excel's formatting strings)
date_format = xlwt.easyxf(num_format_str="mm/dd/yyyy")
sheet.write(4, 0, dt.datetime(2012, 2, 3), date_format)

# Formula: you must use the English name of the formula
# with commas as delimiters
sheet.write(5, 0, xlwt.Formula("SUM(A4, 2)"))

# Two-dimensional list
data = [
    [None, "North", "South"],
    ["Last Year", 2, 5],
    ["This Year", 3, 6],
]
write_xlwt(sheet=sheet, values=data, first_cell="A10")

# Picture (only allows to add bmp format)
sheet.insert_bitmap("images/python.bmp", 0, 2)

# This writes the file to disk
book17.save("xlwt.xls")

### Editing with xlutils


In [ ]:
import xlutils.copy

In [ ]:
book18 = xlrd.open_workbook("xl/stores.xls", formatting_info=True)
book19 = xlutils.copy.copy(book18)
book19.get_sheet(0).write(0, 0, "changed!")
book19.save("stores_edited.xls")

## Formatting DataFrames in Excel


In [ ]:
with pd.ExcelWriter(
    "pandas_and_xlsxwriter.xlsx",
    engine="xlsxwriter",
) as writer:
    df = pd.DataFrame({"col1": [1, 2, 3, 4], "col2": [5, 6, 7, 8]})
    # Write a DataFrame
    df.to_excel(writer, sheet_name="Sheet1", startrow=4, startcol=2)

    # Get the XlsxWriter workbook and sheet objects
    book20 = writer.book
    sheet = writer.sheets["Sheet1"]

    # From here on, it's XlsxWriter code
    sheet.write("A1", "This is a Title")  # Write a single cell value

In [ ]:
df = pd.DataFrame(
    {"col1": [1, -2], "col2": [-3, 4]},
    index=["row1", "row2"],
)
df.index.name = "ix"
df

In [ ]:
# Formatting index/headers with XlsxWriter
with pd.ExcelWriter(
    "formatting_xlsxwriter.xlsx",
    engine="xlsxwriter",
) as writer:
    # Write out the df with the default formatting to A1
    df.to_excel(writer, startrow=0, startcol=0)

    # Write out the df with custom index/header formatting to F1
    startrow, startcol = address_to_indices("F1")
    # 1. Write out the data part of the DataFrame
    df.to_excel(
        writer,
        header=False,
        index=False,
        startrow=startrow + 1,
        startcol=startcol + 1,
    )
    # Get the book and sheet object and create a style object
    book22 = writer.book
    sheet = writer.sheets["Sheet1"]
    style = book22.add_format({"bg_color": "#D9D9D9"})

    # 2. Write out the styled column headers
    for row_ix, col in enumerate(df.columns):
        sheet.write(startrow, startcol + row_ix + 1, col, style)

    # 3. Write out the styled index
    index = [df.index.name if df.index.name else None] + list(df.index)
    for row_ix, row in enumerate(index):
        sheet.write(startrow + row_ix, startcol, row, style)

In [ ]:
with pd.ExcelWriter(
    "formatting_openpyxl.xlsx",
    engine="openpyxl",
) as writer:
    # Write out the df with the default formatting to A1
    df.to_excel(writer, startrow=0, startcol=0)

    # Write out the df with custom index/header formatting to F1
    startrow, startcol = address_to_indices("F1")
    # 1. Write out the data part of the DataFrame
    df.to_excel(
        writer,
        header=False,
        index=False,
        startrow=startrow + 1,
        startcol=startcol + 1,
    )
    # Get the sheet object and create a style object
    sheet = writer.sheets["Sheet1"]
    style = PatternFill(fgColor="D9D9D9", fill_type="solid")

    # 2. Write out the styled column headers
    for row_ix, col in enumerate(df.columns):
        sheet.cell(
            row=startrow + 1,
            column=row_ix + startcol + 2,
            value=col,
        ).fill = style

    # 3. Write out the styled index
    index = [df.index.name if df.index.name else None] + list(df.index)
    for row_ix, row in enumerate(index):
        sheet.cell(
            row=row_ix + startrow + 1,
            column=startcol + 1,
            value=row,
        ).fill = style

In [ ]:
with pd.ExcelWriter(
    "data_format_xlsxwriter.xlsx",
    engine="xlsxwriter",
) as writer:
    # Write out the DataFrame
    df.to_excel(writer)

    # Get the book and sheet objects
    book23 = writer.book
    sheet = writer.sheets["Sheet1"]

    # Formatting the columns (individual cells can't be formatted)
    number_format = book23.add_format(
        {"num_format": "0.000", "align": "center"},
    )
    sheet.set_column(
        first_col=1,
        last_col=2,
        cell_format=number_format,
    )

In [ ]:
from openpyxl.styles import Alignment

In [ ]:
with pd.ExcelWriter(
    "data_format_openpyxl.xlsx",
    engine="openpyxl",
) as writer:
    # Write out the DataFrame
    df.to_excel(writer)

    # Get the book and sheet objects
    book24 = writer.book
    sheet = writer.sheets["Sheet1"]

    # Formatting individual cells
    nrows, ncols = df.shape
    for row in range(nrows):
        for col in range(ncols):
            # +1 to account for the header/index
            # +1 since OpenPyXL is 1-based
            cell = sheet.cell(row=row + 2, column=col + 2)
            cell.number_format = "0.000"
            cell.alignment = Alignment(horizontal="center")

In [ ]:
df = pd.DataFrame(
    {
        "Date": [dt.date(2020, 1, 1)],
        "Datetime": [dt.datetime(2020, 1, 1, 10)],
    }
)
with pd.ExcelWriter(
    "date.xlsx",
    date_format="yyyy-mm-dd",
    datetime_format="yyyy-mm-dd hh:mm:ss",
) as writer:
    df.to_excel(writer)